In [1]:
import zmq
import cloudpickle
import time
import traceback

context = zmq.Context()

socket = context.socket(zmq.REP)
socket.bind("tcp://*:5556")

<SocketContext(bind='tcp://*:5556')>

In [3]:
from qick import *
from qick.averager_program import *
soc = QickSoc()
soccfg = soc
print("Config chargée")

ModuleNotFoundError: No module named 'qick'

In [2]:
while True:
    job = cloudpickle.loads(socket.recv())
    if job["type"] == "get_config":  
        socket.send(cloudpickle.dumps(soccfg.get_cfg()))
        print("config donnée")
        continue
    
    if job["type"] == "STOP":#stop n'est pas encore implémenté
        socket.send(cloudpickle.dumps({"status": "stopped"}))
        print("arret")
        break
        
    prog_class = job["prog"]
    config = job["config"]
    nom = job["nom"]
    print("Traitement :", nom)
    
    try:
        prog = prog_class(soccfg,config)
        if job["acquire_method"]=="acquire_decimated":
            result = prog.acquire_decimated(soc,progress=False)
        elif job["acquire_method"]=="acquire_round":
            result = prog.acquire_round(soc,progress=False)
        elif job["acquire_method"]=="acquire":
            result = prog.acquire(soc,progress=False)
        response = {
        "status": "done",
        "result": result}

    except Exception as e:
        response = {
        "status": "failed",
        "error_type": type(e).__name__,
        "result": traceback.format_exc()}
    socket.send(cloudpickle.dumps(response))
    print(f"Traitement de {nom} terminé")

NameError: name 'soccfg' is not defined